# CHIRPS daily rainfall, 2026-01-01 to latest available day: Kakamega, Kilifi, Nyeri

Daily CHIRPS rainfall (mm/day, ~5.5 km) clipped to each county, then gap-filled.

**Outputs** (under `OUTPUT_DIR = F:\CHIRPS_2026`):
- `county_daily_direct/<County>/CHIRPS_<County>_YYYY_MM_DD.tif` - raw Earth Engine clip (nodata -9999)
- `clipped_daily/<County>/...` - re-clipped with rasterio
- `filled_daily/<County>/...` - **final gap-filled daily rasters**
- `stats_daily/ALL_counties_daily_rainfall.csv`, per-county CSVs and `..._FILLED.csv` (mean rainfall, valid pixels)
- `logs/` - gap-fill and failed-download logs

**Notes**
- The end date is capped at the latest day CHIRPS has published, so future days are never filled with copied data. Re-run any time to pick up new days; existing files are skipped.
- The most recent CHIRPS days are preliminary and get revised later. To refresh them, delete those days' files from `county_daily_direct` and `clipped_daily` and re-run.
- If an earlier run created `filled_daily` files for dates after the latest CHIRPS day, delete them.
- Days download in parallel (`MAX_WORKERS`) with a progress bar. If the bar shows as text, install ipywidgets and restart the kernel.


In [ ]:
# Run once, then restart the kernel if pip installed anything
%pip install geemap geedim geopandas scikit-learn tqdm ipywidgets rasterio scipy matplotlib


In [ ]:
import ee

ee.Authenticate()                      
ee.Initialize(project='poised-list-512309-u8') # your Google Cloud project ID


In [ ]:
import os
import time
import bisect
import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
EE_PROJECT_ID = "thursday-512108"

SHAPEFILE_PATH = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"
NAME_COLUMN = "adm1_name"
COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]  # county-only processing (no Kenya-wide download)

OUTPUT_DIR = r"F:\CHIRPS_2026"
COUNTY_TIF_DIR = os.path.join(OUTPUT_DIR, "county_daily_direct")   # direct EE clip per county
COUNTY_CLIP_DIR = os.path.join(OUTPUT_DIR, "clipped_daily")        # kept for the rasterio re-clip pass
STATS_DIR = os.path.join(OUTPUT_DIR, "stats_daily")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

START_YEAR, END_YEAR = 2026, 2026  # 2026-01-01 up to the latest day CHIRPS has published (capped automatically below)
MAX_WORKERS = 4   # days downloaded in parallel; lower to 2-3 if you see many Earth Engine 429 / quota errors
SCALE_M = 5566  # CHIRPS native resolution in metres (~0.05 deg) — matches EE's nominalScale, do not change

# NODATA sentinel used consistently from the EE export all the way through
# to the final clipped rasters. CHIRPS values are >= 0, so a large negative
# sentinel can never collide with real rainfall data.
NODATA_VALUE = -9999

# If True, at the very end of the run, plot one sample clipped raster per
# county with nodata pixels masked out (transparent), as a quick visual
# check that the AOI clip actually worked.
PLOT_SAMPLE_AFTER_RUN = True

# If True, delete any previously-downloaded per-county EE exports and their
# rasterio-clipped versions before this run, so every file is regenerated
# with the current nodata/clip logic. Turn this on ONCE after changing the
# clip/nodata logic, then set it back to False for normal resumable runs —
# otherwise you re-download everything every time, which is slow and
# wasteful over a full year of daily data.
FORCE_REBUILD_COUNTY_OUTPUTS = False   # True wipes county_daily_direct + clipped_daily and re-downloads everything

# If True, before anything else runs, scan every existing .tif in
# COUNTY_TIF_DIR / COUNTY_CLIP_DIR and force-write the NODATA_VALUE tag
# onto any file that's missing one — this repairs old files left over
# from before the nodata fix existed, without having to redownload them.
# Safe to leave True; it's a no-op once every file already has the tag.
# Does NOT touch pixel values, only the metadata tag — see
# REWRITE_SUSPECTED_ZERO_FILL below if the tag alone doesn't fix a file.
RUN_LEGACY_NODATA_REPAIR = True

# Only relevant if RUN_LEGACY_NODATA_REPAIR is True. If a legacy file has
# NO nodata tag at all, its masked-out pixels were likely written as plain
# 0 instead of a sentinel. Enabling this rewrites those 0-valued pixels to
# NODATA_VALUE before tagging. Leave this False until you've confirmed via
# the printed diagnostics that 0 really is leftover fill and not a genuine
# "0 mm rainfall that day" reading — this can't tell the difference.
REWRITE_SUSPECTED_ZERO_FILL = False

# If True, run the two-tier gap-filling pass (Step 6.5) after clipping:
# tier 1 fills partial within-day gaps via nearest-neighbor spatial
# interpolation; tier 2 fills whole missing/empty days by averaging the
# nearest previous and next REAL (unfilled) day. Leave this True unless
# you specifically want to see the raw, gappy output.
ENABLE_GAP_FILLING = True

COUNTY_FILLED_DIR = os.path.join(OUTPUT_DIR, "filled_daily")  # gap-filled final output
GAP_FILL_LOG = os.path.join(LOG_DIR, "gap_fill_log.csv")

FAILED_DOWNLOADS_LOG = os.path.join(LOG_DIR, "failed_downloads.csv")
FAILED_COUNTY_DOWNLOADS_LOG = os.path.join(LOG_DIR, "failed_county_downloads.csv")
FAILED_CLIPS_LOG = os.path.join(LOG_DIR, "failed_clips.csv")

PROGRESS_EVERY_N_DAYS = 50

for d in [COUNTY_TIF_DIR, COUNTY_CLIP_DIR, COUNTY_FILLED_DIR, STATS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

if FORCE_REBUILD_COUNTY_OUTPUTS:
    import shutil
    print("FORCE_REBUILD_COUNTY_OUTPUTS is True — wiping old county outputs "
          "so everything regenerates with the current clip/nodata logic...")
    for county in COUNTIES:
        for base_dir in [COUNTY_TIF_DIR, COUNTY_CLIP_DIR]:
            county_path = os.path.join(base_dir, county)
            if os.path.isdir(county_path):
                shutil.rmtree(county_path)
                print(f"   Removed {county_path}")

# ============================================================
# STEP 0: LEGACY NODATA REPAIR — fixes files left over from before the
# nodata fix existed, without redownloading them.
# ============================================================
if RUN_LEGACY_NODATA_REPAIR:
    print("0. Repairing legacy files that are missing a nodata tag...")

    def repair_nodata_tag(path, nodata_value, rewrite_zero_fill):
        with rasterio.open(path) as src:
            current_nodata = src.nodata
            if current_nodata == nodata_value:
                return "already_ok"
            data = src.read(1)
            profile = src.profile.copy()

        if rewrite_zero_fill and current_nodata is None:
            data = data.copy()
            data[data == 0] = nodata_value

        profile.update(nodata=nodata_value)
        with rasterio.open(path, "w", **profile) as dst:
            dst.write(data, 1)
        return "fixed"

    _repair_total, _repair_fixed, _repair_errors = 0, 0, []
    for folder in [COUNTY_TIF_DIR, COUNTY_CLIP_DIR]:
        if not os.path.isdir(folder):
            continue
        for root, _, files in os.walk(folder):
            for fname in files:
                if not fname.lower().endswith(".tif"):
                    continue
                _repair_total += 1
                fpath = os.path.join(root, fname)
                try:
                    if repair_nodata_tag(fpath, NODATA_VALUE, REWRITE_SUSPECTED_ZERO_FILL) == "fixed":
                        _repair_fixed += 1
                except Exception as e:
                    _repair_errors.append((fpath, str(e)))

    print(f"   Scanned {_repair_total} existing files, repaired {_repair_fixed} missing nodata tags.")
    if _repair_errors:
        print(f"   {len(_repair_errors)} files errored during repair, e.g.: {_repair_errors[0]}")

# ============================================================
# STEP 1: EARTH ENGINE AUTH
# ============================================================
print("1. Initializing Earth Engine...")
try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)

# ============================================================
# STEP 2: BUILD COUNTY GEOMETRIES
# ============================================================
print("2. Loading county shapefiles...")
_ALL_GDF = None


def load_county_gdf(county):
    """Read ken_admin1.shp once; return the rows whose NAME_COLUMN matches this county (EPSG:4326)."""
    global _ALL_GDF
    if _ALL_GDF is None:
        g = gpd.read_file(SHAPEFILE_PATH)
        g = g.set_crs(epsg=4326) if g.crs is None else g.to_crs(epsg=4326)
        g["_n"] = g[NAME_COLUMN].astype(str).str.title()
        _ALL_GDF = g
    gdf = _ALL_GDF[_ALL_GDF["_n"] == county.title()].drop(columns="_n").copy()
    if gdf.empty:
        raise ValueError(f"County {county!r} not found in column {NAME_COLUMN!r}; available: {sorted(_ALL_GDF['_n'].unique())}")
    if not gdf.geometry.is_valid.all():
        gdf["geometry"] = gdf.geometry.buffer(0)
    return gdf


county_gdfs = {}
county_ee_geoms = {}
for county in COUNTIES:
    gdf = load_county_gdf(county)
    county_gdfs[county] = gdf
    county_union = gdf.union_all()
    county_ee_geoms[county] = ee.Geometry(county_union.__geo_interface__)
    b = gdf.total_bounds
    print(f"   {county}: bounds (lon/lat) = {list(b)}")

# Union of all requested counties' geometries — used only to bound the
# CHIRPS collection query, not for any export (Kakamega alone here).
counties_union_geom = ee.Geometry(list(county_ee_geoms.values())[0])
for g in list(county_ee_geoms.values())[1:]:
    counties_union_geom = counties_union_geom.union(g, maxError=1)

# ============================================================
# STEP 3: CHIRPS DAILY COLLECTION
# ============================================================
print("3. Preparing CHIRPS daily collection reference...")
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(counties_union_geom)

# Cap the range at the latest day CHIRPS has published (and at today), so future days are never "gap-filled" with copied data
_last_ms = chirps.sort("system:time_start", False).first().get("system:time_start").getInfo()
CHIRPS_LATEST = pd.Timestamp(_last_ms, unit="ms").normalize()
_today = pd.Timestamp.now("UTC").tz_localize(None).normalize()
_end = min(pd.Timestamp(f"{END_YEAR}-12-31"), CHIRPS_LATEST, _today)
DATE_RANGE = pd.date_range(f"{START_YEAR}-01-01", _end, freq="D")
print(f"   Latest CHIRPS day available: {CHIRPS_LATEST.date()}  ->  processing {DATE_RANGE[0].date()} to {DATE_RANGE[-1].date()}")
all_dates = DATE_RANGE
n_days = len(all_dates)
print(f"   Total days to process: {n_days}")


def clip_and_unmask(img, geom):
    """
    Clip to geometry AND bake in an explicit NoData sentinel so the
    exported GeoTIFF's pixel VALUES reflect the clip. Note that this alone
    is not enough for GIS software (QGIS/ArcGIS) to hide those pixels —
    see set_nodata_tag() below, which writes the actual GDAL nodata tag
    that EE's download endpoint does not set on its own.
    """
    return (
        img.clip(geom)
        .unmask(NODATA_VALUE, sameFootprint=False)
    )


def set_nodata_tag(path, nodata_value):
    """
    Write the GDAL nodata tag into a GeoTIFF's metadata in-place.

    This is the piece that was missing: geemap.ee_export_image() downloads
    whatever pixel values Earth Engine computed (including the -9999
    sentinel from clip_and_unmask), but it does NOT write a `nodata` tag
    into the GeoTIFF metadata just because the image was unmask()'d on the
    EE side. Without that tag, QGIS/ArcGIS have no way to know -9999 means
    "nothing here" and will render it as real data — which is exactly why
    areas outside the polygon were showing up in the exported .tif files.

    Called right after every geemap.ee_export_image() call, so every fresh
    file gets the tag immediately rather than depending on the next run's
    legacy-repair pass (which only fixes files that already existed when
    the script started, not ones created later in the same run).
    """
    with rasterio.open(path, "r+") as dst:
        dst.nodata = nodata_value


# ============================================================
# STEP 4: DOWNLOAD — DIRECT PER-COUNTY (all counties)
# ============================================================
print("4. Downloading daily rasters for all counties...")
failed_downloads = []
failed_county_downloads = []
start_time = time.time()

def export_with_retry(img, out_path, geom, tries=3):
    for k in range(1, tries + 1):
        try:
            geemap.ee_export_image(img, filename=out_path, scale=SCALE_M, region=geom,
                                   crs="EPSG:4326", file_per_band=False)
            return
        except Exception:
            if k == tries:
                raise
            time.sleep(3 * k)


def fetch_one_date(d):
    year, month, day = d.year, d.month, d.day
    date_str = d.strftime("%Y-%m-%d")
    next_day_str = (d + datetime.timedelta(days=1)).strftime("%Y-%m-%d")
    county_out_paths = {
        county: os.path.join(COUNTY_TIF_DIR, county, f"CHIRPS_{county}_{year}_{month:02d}_{day:02d}.tif")
        for county in COUNTIES
    }
    counties_needed = [c for c in COUNTIES if not os.path.exists(county_out_paths[c])]
    if not counties_needed:
        return  # everything for this date already downloaded

    try:
        day_img = chirps.filterDate(date_str, next_day_str).first()
        band_names = day_img.bandNames().getInfo()
        if not band_names:
            raise ValueError("No CHIRPS image found for this date")
    except Exception as e:
        tqdm.write(f"   Failed to fetch CHIRPS image for {date_str}: {e}")
        failed_downloads.append({"date": date_str, "error": str(e)})
        for county in counties_needed:
            failed_county_downloads.append({"county": county, "date": date_str, "error": str(e)})
        return

    for county in counties_needed:
        out_path = county_out_paths[county]
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        try:
            geom = county_ee_geoms[county]
            img = clip_and_unmask(day_img, geom).set("year", year, "month", month, "day", day)
            export_with_retry(img, out_path, geom)
            set_nodata_tag(out_path, NODATA_VALUE)
        except Exception as e:
            tqdm.write(f"   Failed {county} download {date_str}: {e}")
            failed_county_downloads.append({"county": county, "date": date_str, "error": str(e)})


# days are downloaded in parallel; the progress bar (ipywidgets) advances as each day finishes
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_one_date, d) for d in all_dates]
    for f in tqdm(as_completed(futures), total=len(futures), desc="CHIRPS download", unit="day"):
        try:
            f.result()
        except Exception as e:
            tqdm.write(f"   Unexpected error: {e}")

if failed_downloads:
    pd.DataFrame(failed_downloads).to_csv(FAILED_DOWNLOADS_LOG, index=False)
    print(f"   {len(failed_downloads)} dates had no CHIRPS image available — see {FAILED_DOWNLOADS_LOG}")
if failed_county_downloads:
    pd.DataFrame(failed_county_downloads).to_csv(FAILED_COUNTY_DOWNLOADS_LOG, index=False)
    print(f"   {len(failed_county_downloads)} county downloads failed — see {FAILED_COUNTY_DOWNLOADS_LOG}")

print("   Re-run this script to retry only the missing files (existing files are skipped).")

# ============================================================
# STEP 5: SANITY-CHECK ONE COUNTY FILE (catches nodata problems early)
# ============================================================
print("5. Spot-checking nodata handling on one exported file...")
for county in COUNTIES:
    county_dir = os.path.join(COUNTY_TIF_DIR, county)
    sample_files = sorted(os.listdir(county_dir)) if os.path.isdir(county_dir) else []
    if sample_files:
        sample_path = os.path.join(county_dir, sample_files[0])
        with rasterio.open(sample_path) as src:
            data = src.read(1)
            print(f"   {county} sample ({sample_files[0]}): "
                  f"nodata tag = {src.nodata}, "
                  f"min/max = {data.min()}/{data.max()}, "
                  f"pixels == {NODATA_VALUE}: {(data == NODATA_VALUE).sum()} / {data.size}")
        break

# ============================================================
# STEP 6: FINAL RASTERIO CLIP + DAILY STATS (per county)
# ============================================================
print("6. Clipping (rasterio, belt-and-braces) and computing daily stats...")


def clip_raster_to_county(raster_path, county_gdf, out_path):
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        geom = county_gdf.to_crs(raster_crs) if county_gdf.crs != raster_crs else county_gdf
        geoms = [feat["geometry"] for feat in geom.__geo_interface__["features"]]
        out_image, out_transform = rio_mask(src, geoms, crop=True, nodata=NODATA_VALUE)
        out_meta = src.meta.copy()
        out_meta.update(
            height=out_image.shape[1],
            width=out_image.shape[2],
            transform=out_transform,
            nodata=NODATA_VALUE,
        )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_image)
    data = out_image[0].astype("float64")
    data[data == NODATA_VALUE] = np.nan
    return data


def compute_stats(data_array):
    valid = data_array[~np.isnan(data_array)]
    if valid.size == 0:
        return dict(mean=np.nan, valid_pixels=0)
    return dict(mean=float(np.mean(valid)), valid_pixels=int(valid.size))


all_records = []
failed_clips = []

for county in COUNTIES:
    county_src_dir = os.path.join(COUNTY_TIF_DIR, county)
    if not os.path.isdir(county_src_dir):
        continue
    tif_files = sorted(f for f in os.listdir(county_src_dir) if f.endswith(".tif"))

    for j, fname in enumerate(tif_files):
        parts = fname.replace(".tif", "").split("_")  # CHIRPS, county, year, month, day
        year, month, day = int(parts[-3]), int(parts[-2]), int(parts[-1])
        raster_path = os.path.join(county_src_dir, fname)
        out_path = os.path.join(
            COUNTY_CLIP_DIR, county, f"CHIRPS_{county}_{year}_{month:02d}_{day:02d}.tif"
        )
        try:
            data = clip_raster_to_county(raster_path, county_gdfs[county], out_path)
            stats = compute_stats(data)
            stats.update({"county": county, "year": year, "month": month, "day": day})
            all_records.append(stats)
        except Exception as e:
            print(f"   Failed {county} {year}-{month:02d}-{day:02d}: {e}")
            failed_clips.append({"county": county, "year": year, "month": month, "day": day, "error": str(e)})

        if (j + 1) % PROGRESS_EVERY_N_DAYS == 0:
            print(f"   {county}: clipped {j + 1}/{len(tif_files)} days")

if failed_clips:
    pd.DataFrame(failed_clips).to_csv(FAILED_CLIPS_LOG, index=False)
    print(f"   {len(failed_clips)} county-day clips failed — see {FAILED_CLIPS_LOG}")

df = pd.DataFrame(all_records)
if not df.empty:
    df["date"] = pd.to_datetime(
        df["year"].astype(str) + "-" + df["month"].astype(str) + "-" + df["day"].astype(str)
    )
    df = df[["date", "county", "year", "month", "day", "mean", "valid_pixels"]]
    df = df.sort_values(["county", "date"]).reset_index(drop=True)

    df.to_csv(os.path.join(STATS_DIR, "ALL_counties_daily_rainfall.csv"), index=False)
    for county in COUNTIES:
        df[df["county"] == county].to_csv(
            os.path.join(STATS_DIR, f"{county}_daily_rainfall.csv"), index=False
        )
    print(f"   Daily stats saved to: {STATS_DIR}")
else:
    print("   No stats computed — check COUNTY_TIF_DIR contents.")

# ============================================================
# STEP 6.5: GAP FILLING — ensure every calendar day exists and every
# pixel inside the county boundary has a value. Two tiers:
#
#   TIER 1 (spatial, same day): if a day has SOME valid pixels inside
#   the county but a few nodata pixels (a partial gap in that CHIRPS
#   scene, or a partially failed export), fill those pixels from the
#   nearest valid pixel on the SAME day's raster. This always succeeds
#   as long as at least one valid AOI pixel exists that day.
#
#   TIER 2 (temporal, adjacent real days): only used when a day is
#   either completely missing (no file — e.g. Earth Engine had no
#   image for that date) or came back fully nodata everywhere inside
#   the county (tier 1 had nothing to interpolate from). Filled using
#   the average of the nearest REAL previous and next day (never
#   averaging two already-filled days together, to avoid compounding
#   guesses); if only one side has a real day, that one is used alone.
#
# Every fill is logged to GAP_FILL_LOG so real vs. filled values stay
# traceable. Output goes to COUNTY_FILLED_DIR — this is the "no empty
# days, no empty pixels" version you should use downstream.
# ============================================================
if ENABLE_GAP_FILLING:
    print("6.5 Gap-filling missing days and missing pixels...")

    def nearest_fill(data, invalid_mask, aoi_mask):
        """Fill invalid_mask&aoi_mask pixels from the nearest valid pixel
        in the same array. Returns (filled_array, was_fixed_mask). If
        there is no valid pixel anywhere in the AOI, nothing is fixed —
        that's left for tier 2."""
        fillable = invalid_mask & aoi_mask
        if not fillable.any():
            return data, np.zeros_like(fillable)
        valid = aoi_mask & ~invalid_mask
        if not valid.any():
            return data, np.zeros_like(fillable)
        idx = distance_transform_edt(~valid, return_distances=False, return_indices=True)
        filled = data.copy()
        filled[fillable] = data[tuple(ind[fillable] for ind in idx)]
        return filled, fillable

    gap_fill_records = []

    for county in COUNTIES:
        clip_dir = os.path.join(COUNTY_CLIP_DIR, county)
        filled_dir = os.path.join(COUNTY_FILLED_DIR, county)
        os.makedirs(filled_dir, exist_ok=True)

        date_range = DATE_RANGE

        # Reference grid (profile/transform/shape) from the first file we can find.
        ref_path = next(
            (
                os.path.join(clip_dir, f"CHIRPS_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif")
                for dt in date_range
                if os.path.exists(os.path.join(clip_dir, f"CHIRPS_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif"))
            ),
            None,
        )
        if ref_path is None:
            print(f"   No clipped files found for {county} in this date range — skipping gap-fill.")
            continue

        with rasterio.open(ref_path) as src:
            profile = src.profile.copy()
            transform = src.transform
            out_shape = (src.height, src.width)

        # AOI mask: True where the pixel is genuinely inside the county
        # polygon (only these pixels ever get filled; outside-AOI pixels
        # stay nodata forever, same as before).
        county_geoms = list(county_gdfs[county].to_crs(profile["crs"]).geometry)
        aoi_mask = ~geometry_mask(county_geoms, out_shape=out_shape, transform=transform, invert=False)

        # Pass 1: tier-1 spatial fill on every day we actually have a file for.
        arrays = {}       # date -> filled array (only set for days with a real file)
        tier1_pixels = {} # date -> count of pixels fixed by tier 1
        n_missing_files = 0
        n_fully_nodata = 0

        for dt in date_range:
            path = os.path.join(clip_dir, f"CHIRPS_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif")
            if not os.path.exists(path):
                n_missing_files += 1
                continue  # whole day missing -> tier 2 territory, handled below
            with rasterio.open(path) as src:
                data = src.read(1).astype("float64")
                nodata = src.nodata if src.nodata is not None else NODATA_VALUE
            invalid = (data == nodata) | np.isnan(data)
            if (invalid & aoi_mask).sum() == int(aoi_mask.sum()):
                n_fully_nodata += 1
                continue  # entire AOI was nodata this day -> treat as missing, goes to tier 2
            filled, fixed = nearest_fill(data, invalid, aoi_mask)
            n_fixed = int(fixed.sum())
            if n_fixed:
                tier1_pixels[dt] = n_fixed
                gap_fill_records.append(
                    {"county": county, "date": dt.date(), "tier": "spatial", "pixels_filled": n_fixed}
                )
            arrays[dt] = filled  # a genuine, usable "real" day (possibly tier-1 patched)

        n_real_days = len(arrays)
        print(f"   {county}: {n_real_days}/{len(date_range)} days had usable real data "
              f"({n_missing_files} had no file at all, {n_fully_nodata} existed but were fully nodata)")

        # Pass 2: tier-2 temporal fill for every date with no entry in `arrays`.
        anchor_dates = sorted(arrays.keys())
        anchor_keys = [d.toordinal() for d in anchor_dates]
        n_unfillable = 0

        for dt in date_range:
            if dt in arrays:
                continue  # already a real (or tier-1-patched) day

            pos = bisect.bisect_left(anchor_keys, dt.toordinal())
            prev_dt = anchor_dates[pos - 1] if pos > 0 else None
            next_dt = anchor_dates[pos] if pos < len(anchor_dates) else None

            if prev_dt is not None and next_dt is not None:
                fill_arr = np.nanmean(np.stack([arrays[prev_dt], arrays[next_dt]]), axis=0)
                source = f"avg({prev_dt.date()},{next_dt.date()})"
            elif prev_dt is not None:
                fill_arr = arrays[prev_dt]
                source = f"prev({prev_dt.date()})"
            elif next_dt is not None:
                fill_arr = arrays[next_dt]
                source = f"next({next_dt.date()})"
            else:
                print(f"   WARNING: {county} {dt.date()} has NO real day anywhere in range to fill from — skipped.")
                gap_fill_records.append(
                    {"county": county, "date": dt.date(), "tier": "UNFILLABLE", "pixels_filled": 0}
                )
                n_unfillable += 1
                continue

            n_filled = int(aoi_mask.sum())
            gap_fill_records.append(
                {"county": county, "date": dt.date(), "tier": f"temporal:{source}", "pixels_filled": n_filled}
            )
            # Store it separately from `arrays` conceptually, but for writing
            # purposes we just need the array — tier-2 days are written below
            # from this dict, not merged back into `arrays` (so later tier-2
            # days never borrow from an already-filled day).
            arrays[dt] = fill_arr

        if n_unfillable:
            print(f"   {county}: {n_unfillable} day(s) could not be filled at all (no real data anywhere in range).")

        # Write every day (real, tier-1, and tier-2) to the final filled dir.
        out_profile = profile.copy()
        out_profile.update(nodata=NODATA_VALUE)
        for dt, arr in arrays.items():
            out_path = os.path.join(filled_dir, f"CHIRPS_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif")
            write_arr = arr.copy()
            write_arr[~aoi_mask] = NODATA_VALUE
            write_arr[np.isnan(write_arr)] = NODATA_VALUE  # true unfillable leftovers
            with rasterio.open(out_path, "w", **out_profile) as dst:
                dst.write(write_arr.astype(out_profile["dtype"]), 1)
        print(f"   {county}: wrote {len(arrays)}/{len(date_range)} days to {filled_dir}")

    if gap_fill_records:
        pd.DataFrame(gap_fill_records).sort_values(["county", "date"]).to_csv(GAP_FILL_LOG, index=False)
        print(f"   Gap-fill log written to {GAP_FILL_LOG}")

    # Recompute daily stats from the gap-filled rasters — this is the
    # complete, no-missing-days/no-missing-pixels version.
    filled_records = []
    for county in COUNTIES:
        filled_dir = os.path.join(COUNTY_FILLED_DIR, county)
        if not os.path.isdir(filled_dir):
            continue
        for fname in sorted(os.listdir(filled_dir)):
            if not fname.endswith(".tif"):
                continue
            parts = fname.replace(".tif", "").split("_")
            year, month, day = int(parts[-3]), int(parts[-2]), int(parts[-1])
            with rasterio.open(os.path.join(filled_dir, fname)) as src:
                data = src.read(1).astype("float64")
                nodata = src.nodata
            valid = data[data != nodata]
            filled_records.append({
                "county": county, "year": year, "month": month, "day": day,
                "mean": float(np.mean(valid)) if valid.size else np.nan,
                "valid_pixels": int(valid.size),
            })

    filled_df = pd.DataFrame(filled_records)
    if not filled_df.empty:
        filled_df["date"] = pd.to_datetime(
            filled_df["year"].astype(str) + "-" + filled_df["month"].astype(str) + "-" + filled_df["day"].astype(str)
        )
        filled_df = filled_df[["date", "county", "year", "month", "day", "mean", "valid_pixels"]]
        filled_df = filled_df.sort_values(["county", "date"]).reset_index(drop=True)
        filled_df.to_csv(os.path.join(STATS_DIR, "ALL_counties_daily_rainfall_FILLED.csv"), index=False)
        for county in COUNTIES:
            filled_df[filled_df["county"] == county].to_csv(
                os.path.join(STATS_DIR, f"{county}_daily_rainfall_FILLED.csv"), index=False
            )
        n_expected = len(DATE_RANGE) * len(COUNTIES)
        print(f"   Filled stats saved to: {STATS_DIR} ({len(filled_df)}/{n_expected} county-days present)")

# ============================================================
# STEP 7: VISUAL QA — PLOT ONE CLIPPED SAMPLE PER COUNTY
# ============================================================
# This is the fix for the red-block issue: nodata pixels (-9999) are
# masked out before plotting instead of being colored by the colormap,
# and the colormap is told to render masked ("bad") pixels as transparent.


def load_masked(raster_path, fallback_nodata=NODATA_VALUE):
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype("float64")
        nodata = src.nodata

    print(f"   -> dtype in file: {data.dtype}, nodata tag in file: {nodata}")
    if nodata is None:
        print(f"   -> WARNING: this file has NO nodata tag set. It was almost "
              f"certainly exported before the fix, or by a different script. "
              f"Falling back to {fallback_nodata}, but if that value doesn't "
              f"actually match what's in the pixels, masking will do nothing. "
              f"Delete this file (or set FORCE_REBUILD_COUNTY_OUTPUTS = True) "
              f"and re-run so it gets regenerated correctly.")
        nodata = fallback_nodata

    unique_low = np.unique(data)[:5]
    print(f"   -> lowest 5 unique pixel values in file: {unique_low}")
    n_matching = int((data == nodata).sum())
    print(f"   -> pixels equal to nodata ({nodata}): {n_matching} / {data.size}")
    if n_matching == 0:
        print(f"   -> WARNING: zero pixels match nodata={nodata}. This file "
              f"was not clipped/unmasked the way this script expects — "
              f"nothing will be masked out. Check which script actually "
              f"produced this file.")

    masked = np.ma.masked_equal(data, nodata)
    return masked, nodata


def plot_masked(masked, title):
    cmap = plt.cm.Blues.copy()
    cmap.set_bad(color="none")  # nodata pixels -> transparent, not colored

    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(masked, cmap=cmap)
    ax.axis("off")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Rainfall (mm)", shrink=0.8)
    plt.tight_layout()
    plt.show()


if PLOT_SAMPLE_AFTER_RUN:
    print("7. Plotting sample rasters for visual QA...")

    for county in COUNTIES:
        # Plot from the gap-filled output when available (that's the
        # complete, deliverable version); fall back to the raw clipped
        # dir if gap-filling was turned off.
        plot_source_dir = (
            os.path.join(COUNTY_FILLED_DIR, county)
            if ENABLE_GAP_FILLING
            else os.path.join(COUNTY_CLIP_DIR, county)
        )
        sample_files = sorted(os.listdir(plot_source_dir)) if os.path.isdir(plot_source_dir) else []
        if not sample_files:
            print(f"   No files found for {county} yet — skipping plot.")
            continue
        sample_path = os.path.join(plot_source_dir, sample_files[0])
        print(f"   {county} sample: {sample_files[0]}")
        masked, nodata = load_masked(sample_path)
        plot_masked(masked, title=f"{county} (gap-filled) — {sample_files[0]}")

print("\nDone. Full daily CHIRPS pipeline complete.")